# ORCA Cube Reorientation — Colab Training (Four Vectors)

Runs PPO training from the **private** `fv-orca-hand-rl` repo and logs metrics
and the trained model to **Weights & Biases**. No Google Drive, no zip uploads.

## Before you run anything — add two Colab Secrets
Click the **🔑 (key) icon** in the left sidebar and add:
- `GITHUB_TOKEN` — a GitHub Personal Access Token with **read** access to the repo
  (GitHub → Settings → Developer settings → Fine-grained tokens). Toggle *Notebook access* on.
- `WANDB_API_KEY` — from https://wandb.ai/authorize. Toggle *Notebook access* on.

Storing them as secrets keeps the keys out of the notebook itself (so they never
get pushed to GitHub or saved to Drive).

> Note: this script trains on **CPU** by design (it's an MLP policy), so a GPU
> runtime is optional. Free Colab disconnects on long runs — use it for quick
> experiments and run the full 20M locally.

In [ ]:
# 1. Clone the private repo (auth via the GITHUB_TOKEN secret)
from google.colab import userdata

REPO_SLUG = "OWNER/fv-orca-hand-rl"   # <-- EDIT: replace OWNER with your GitHub username or org

token = userdata.get("GITHUB_TOKEN")
name = REPO_SLUG.split("/")[1]
!rm -rf /content/{name}
!git clone https://{token}@github.com/{REPO_SLUG}.git /content/{name}
%cd /content/{name}

In [ ]:
# 2. Install dependencies (this also pulls orca_sim from source). Takes a few minutes.
!pip install -q -r requirements.txt

In [ ]:
# 3. Log in to Weights & Biases (auth via the WANDB_API_KEY secret)
import os
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
import wandb
wandb.login()

In [ ]:
# 4. Smoke test — confirm the whole pipeline runs end to end (a few minutes)
!python train.py --timesteps 20000 --run-name colab-smoke-test

Open the W&B run link printed above — you should see live curves and, when it
finishes, a `model-<run_id>` artifact. Once the smoke test works, you can launch
a longer run below.

For a full 20M-step run, prefer your local machine (no session limits). If you
do run it here and get disconnected, the periodic W&B model uploads mean you
won't lose everything — pull the latest `model-<run_id>` artifact and resume.

In [ ]:
# 5. (Optional) Longer run. Adjust --timesteps as needed.
# !python train.py --timesteps 20000000 --run-name colab-prod-20M